# Лабораторная работа 1 — рабочий ноутбук

**ФИО:** Гончаренок Андрей Валерьевич
**Файлы данных:** clients.csv , transactions.csv

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir
clients_path = os.path.join(project_root, 'data', 'clients.csv')
txn_path = os.path.join(project_root, 'data', 'transactions.csv')

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [ ]:
try:
    df_clients = pd.read_csv(clients_path, sep=',', encoding='utf-8')
    print(f"\nЗагружено {df_clients.shape[0]} строк и {df_clients.shape[1]} столбцов.")
except UnicodeDecodeError:
    print("\n Ошибка кодировки utf-8.")
    try:
        df_clients = pd.read_csv(clients_path, sep=',', encoding='cp1251')
        print(f"C cp1251! Загружено {df_clients.shape[0]} строк.")
    except Exception as e:
        print(f"Всё равно ошибка: {e}")
except Exception as e:
    print(f"Ошибка при чтении: {e}")
    
if 'df_clients' in locals():
    print("\n--- Первые 5 строки clients.csv ---")
    display(df_clients.head(5))

In [ ]:
try:
    df_transactions = pd.read_csv(
        txn_path,
        sep=';',
        encoding='cp1251',
        parse_dates=['txn_datetime'],
        na_values=['', 'NA', 'NaN', 'н/д', '-', 'NULL']
    )
    print(f"transactions.csv успешно загружен: {df_transactions.shape[0]} строк, {df_transactions.shape[1]} столбцов")
except Exception as e:
    print(f"Ошибка загрузки transactions.csv: {e}")

if 'df_transactions' in locals():
    print("\n--- Первые 5 строки transactions.csv ---")
    display(df_transactions.head(5))

## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [ ]:
print("=" * 60)
print("CLIENTS.CSV")
print("=" * 60)
print(f"Размер: {df_clients.shape[0]} строк, {df_clients.shape[1]} столбцов")
print(f"\nЕдиница наблюдения: одна строка = один клиент банка")
print(f"Главный ключ: client_id")

print("\n--- Типы данных, уникальные значения и пропуски ---")
overview_clients = pd.DataFrame({
    'dtype': df_clients.dtypes.astype(str),
    'n_unique': df_clients.nunique(),
    'n_missing': df_clients.isna().sum(),
    'missing_%': (df_clients.isna().mean() * 100).round(2),
})
display(overview_clients)

print("\n" + "=" * 60)
print("TRANSACTIONS.CSV")
print("=" * 60)
print(f"Размер: {df_transactions.shape[0]} строк, {df_transactions.shape[1]} столбцов")
print(f"\nЕдиница наблюдения: одна строка = одна транзакция")
print(f"Главный ключ: txn_id")

print("\n--- Типы данных, уникальные значения и пропуски ---")
overview_txn = pd.DataFrame({
    'dtype': df_transactions.dtypes.astype(str),
    'n_unique': df_transactions.nunique(),
    'n_missing': df_transactions.isna().sum(),
    'missing_%': (df_transactions.isna().mean() * 100).round(2),
})
display(overview_txn)

### 2.2. Дубли

In [ ]:
print("=" * 60)
print("ПРОВЕРКА ДУБЛИКАТОВ")
print("=" * 60)

# --- clients ---
print("\n--- clients.csv ---")
print(f"Всего строк: {len(df_clients)}")
print(f"Уникальных client_id: {df_clients['client_id'].nunique()}")
print(f"Дубликатов client_id: {df_clients['client_id'].duplicated().sum()}")
print(f"Полных дубликатов строк (все столбцы совпадают): {df_clients.duplicated().sum()}")

# Примеры дубликатов
client_dups = df_clients[df_clients['client_id'].duplicated(keep=False)]
if len(client_dups) > 0:
    print(f"\nПримеры дубликатов (первые 10 строк):")
    display(client_dups.sort_values('client_id').head(10))

# --- transactions ---
print("\n--- transactions.csv ---")
print(f"Всего строк: {len(df_transactions)}")
print(f"Уникальных txn_id: {df_transactions['txn_id'].nunique()}")
print(f"Дубликатов txn_id: {df_transactions['txn_id'].duplicated().sum()}")
print(f"Полных дубликатов строк: {df_transactions.duplicated().sum()}")

txn_dups = df_transactions[df_transactions['txn_id'].duplicated(keep=False)]
if len(txn_dups) > 0:
    print(f"\nПримеры дубликатов (первые 10 строк):")
    display(txn_dups.sort_values('txn_id').head(10))

### 2.3. Соответствие данных описанию полей из задания

In [ ]:
# ============================================================
# 2.3. Соответствие данных описанию полей из задания
# ============================================================
# Описание полей взято из TASK.md.
# Расхождение между описанием и фактом фиксируется как находка.

print("=" * 70)
print("ПРОВЕРКА СООТВЕТСТВИЯ ДАННЫХ ОПИСАНИЮ ИЗ TASK.md")
print("=" * 70)

# ============================================================
# transactions.csv
# ============================================================
print("\n--- transactions.csv ---")

# Проверка category на соответствие описанию "одна из девяти"
print("\n1. Поле 'category' (в описании: 'категория траты, одна из девяти'):")
n_cat = df_transactions['category'].nunique()
print(f"   Фактическое количество уникальных значений: {n_cat}")
print(f"   Значения: {df_transactions['category'].unique()}")
if n_cat != 9:
    print(f"   РАСХОЖДЕНИЕ: в описании указано 9, фактически найдено {n_cat} категорий.")
else:
    print("   Соответствует описанию.")

# Проверка channel на соответствие описанию (4 значения)
print("\n2. Поле 'channel' (в описании: 'Карта, Токен, СБП, Онлайн-банк'):")
n_ch = df_transactions['channel'].nunique()
print(f"   Фактическое количество уникальных значений: {n_ch}")
print(f"   Значения: {df_transactions['channel'].unique()}")
if n_ch == 4:
    print("   Соответствует описанию.")
else:
    print(f"   РАСХОЖДЕНИЕ: ожидалось 4 значения, найдено {n_ch}.")

# Проверка status на соответствие описанию (3 значения)
print("\n3. Поле 'status' (в описании: 'успешно, отклонено, возврат'):")
n_st = df_transactions['status'].nunique()
print(f"   Фактическое количество уникальных значений: {n_st}")
print(f"   Значения: {df_transactions['status'].unique()}")
if n_st == 3:
    print("   Соответствует описанию.")
else:
    print(f"   РАСХОЖДЕНИЕ: ожидалось 3 значения, найдено {n_st}.")

# ============================================================
# clients.csv
# ============================================================
print("\n--- clients.csv ---")

# Проверка satisfaction_score на диапазон [1, 5]
print("\n4. Поле 'satisfaction_score' (в описании: 'оценка от 1 до 5'):")
print(f"   Минимум: {df_clients['satisfaction_score'].min()}")
print(f"   Максимум: {df_clients['satisfaction_score'].max()}")
if df_clients['satisfaction_score'].min() < 1 or df_clients['satisfaction_score'].max() > 5:
    print("   НАРУШЕНИЕ ВАЛИДНОСТИ: значения выходят за диапазон [1, 5].")
else:
    print("   Соответствует описанию.")

# Проверка branch_code на неотрицательность
print("\n5. Поле 'branch_code' (в описании: 'код отделения обслуживания'):")
print(f"   Минимум: {df_clients['branch_code'].min()}")
print(f"   Максимум: {df_clients['branch_code'].max()}")
if df_clients['branch_code'].min() < 0:
    print("   НАРУШЕНИЕ ВАЛИДНОСТИ: найдены отрицательные коды.")
else:
    print("   Соответствует описанию.")

# Проверка monthly_income на скрытые пропуски (отрицательные значения)
print("\n6. Поле 'monthly_income' (в описании: 'ежемесячный доход, рублей'):")
min_income = df_clients['monthly_income'].min()
print(f"   Минимальное значение: {min_income}")
if min_income < 0:
    print(f"   НАРУШЕНИЕ ВАЛИДНОСТИ: найдено отрицательное значение {min_income} (вероятно, технический код пропуска).")

# Проверка city на согласованность регистра
print("\n7. Поле 'city' (в описании: 'город проживания'):")
n_cities = df_clients['city'].nunique()
n_cities_lower = df_clients['city'].str.lower().nunique()
print(f"   Уникальных значений с учётом регистра: {n_cities}")
print(f"   Уникальных значений без учёта регистра: {n_cities_lower}")
if n_cities > n_cities_lower:
    print(f"   НАРУШЕНИЕ СОГЛАСОВАННОСТИ: есть города с разным регистром (например, 'Москва' и 'москва').")
    # Примеры пар с разным регистром
    city_counts = df_clients['city'].str.lower().value_counts()
    ambiguous = city_counts[city_counts > 1].index.tolist()
    print(f"   Города, встречающиеся в разных регистрах: {ambiguous[:5]}")
else:
    print("   Согласованность регистра соблюдена.")

# ============================================================
# Итог
# ============================================================
print("\n" + "=" * 70)
print("ИТОГОВЫЙ СПИСОК РАСХОЖДЕНИЙ")
print("=" * 70)
print("""
1. transactions.csv, поле 'category': описание указывает 9 категорий,
   фактически найдено 12 уникальных значений.

2. clients.csv, поле 'monthly_income': найдено значение -999,
   что является скрытым пропуском (доход не может быть отрицательным).

3. clients.csv, поле 'city': нарушена согласованность регистра
   (например, 'Москва' и 'москва'), что завышает число уникальных значений.
""")

### 2.4. Связь между таблицами

In [ ]:
print("=" * 60)
print("СВЯЗЬ МЕЖДУ TABLES")
print("=" * 60)

# все ли client_id из transactions есть в clients
clients_ids = set(df_clients['client_id'].unique())
txn_ids = set(df_transactions['client_id'].unique())

print(f"\nУникальных клиентов в clients.csv: {len(clients_ids)}")
print(f"Уникальных клиентов в transactions.csv: {len(txn_ids)}")

# Клиенты из транзакций, которых нет в справочнике клиентов
missing_in_clients = txn_ids - clients_ids
print(f"\nКлиентов в транзакциях, но НЕ в clients.csv: {len(missing_in_clients)}")
if len(missing_in_clients) > 0:
    print(f"Примеры: {list(missing_in_clients)[:10]}")

# Клиенты из справочника без транзакций
missing_in_txn = clients_ids - txn_ids
print(f"\nКлиентов в clients.csv, но БЕЗ транзакций: {len(missing_in_txn)}")
if len(missing_in_txn) > 0:
    print(f"Примеры: {list(missing_in_txn)[:10]}")

# Кардинальность связи
print("\n--- Кардинальность связи ---")
txn_per_client = df_transactions.groupby('client_id').size()
print(f"Транзакций на клиента:")
print(f"  Минимум: {txn_per_client.min()}")
print(f"  Максимум: {txn_per_client.max()}")
print(f"  Медиана: {txn_per_client.median()}")
print(f"  Среднее: {txn_per_client.mean():.1f}")

# Распределение числа транзакций
print(f"\nРаспределение числа транзакций на клиента:")
print(txn_per_client.describe())

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

# Выводы по аудиту качества данных

## 1. clients.csv (5727 строк × 16 столбцов)

### 1.1. Дубликаты
Обнаружено **55 дубликатов** по полю `client_id` (из 5727 строк уникальных ID — 5672). Из них **41 строка** является полным дубликатом (совпадают все 16 полей). Это нарушает критерий уникальности.

### 1.2. Невалидные значения и скрытые пропуски
- `monthly_income`: минимум **-999.0**. Это не реальный доход, а технический код пропуска (скрытый пропуск), который необходимо заменить на `NaN`.
- `satisfaction_score` и `branch_code`: диапазоны корректны (1.0–5.0 и 100–939 соответственно), невалидных значений не обнаружено.

### 1.3. Несоответствие типов
- Колонки `birth_date`, `registration_date`, `card_open_date` имеют тип `str` (object) вместо ожидаемого `datetime64`. Это делает невозможным расчёт возраста клиента и стажа.

### 1.4. Проблемы согласованности
- В поле `city` встречаются значения с разным регистром (например, `'Москва'` и `'москва'`). Это искусственно раздувает число уникальных значений.

### 1.5. Пропуски
| Поле | Пропусков | Доля | Вероятный механизм |
|------|-----------|------|---------------------|
| `monthly_income` | 559 | 9.76% | MAR (не указывают клиенты с нестабильным доходом) |
| `app_last_login` | 2187 | 38.19% | MAR (отсутствует у клиентов без приложения) |
| `satisfaction_score` | 3463 | 60.47% | MAR/MNAR (опрос проходят только активные клиенты) |

---

## 2. transactions.csv (625 787 строк × 9 столбцов)

### 2.1. Дубликаты
Обнаружено **1500 дубликатов** по полю `txn_id` (из 625 787 строк уникальных — 624 287). Из них **1284 строки** — полные дубликаты.

### 2.2. Несоответствие типов
| Поле | Текущий тип | Ожидаемый | Проблема |
|------|-------------|-----------|----------|
| `txn_id` | `int64` | `string` | Риск потери ведущих нулей, ошибки при merge |
| `txn_datetime` | `str` | `datetime64` | Невозможны временные операции |
| `amount` | `str` | `float64` | Невозможны математические операции (суммирование) |
| `category`, `channel`, `status`, `currency`, `mcc_code` | `str` / `int64` | `category` | Неэффективно по памяти (мало уникальных значений) |

### 2.3. Расхождения с описанием из TASK.md и нарушения валидности

При сверке фактических данных с описанием из `TASK.md` выявлены следующие расхождения:

1. **transactions.csv, поле `category`**: в описании указано "одна из девяти" категорий, но фактически в данных присутствует **12 уникальных значений**. Это требует дополнительного маппинга или уточнения у владельца данных.
2. **clients.csv, поле `monthly_income`**: обнаружено отрицательное значение (минимум **-999**). Доход не может быть отрицательным, следовательно, это **скрытый пропуск** (технический код), который необходимо заменить на `NaN`.
3. **clients.csv, поле `city`**: нарушена согласованность формата. Встречаются названия городов с разным регистром (например, `'Москва'` и `'москва'`), что искусственно завышает количество уникальных значений.

*Примечание: поля `channel` и `status` в транзакциях полностью соответствуют описанию (4 и 3 уникальных значения соответственно).*

### 2.4. Пропуски
Отсутствуют (полнота 100%), что соответствует ожиданиям для процессинговой системы.

---

## 3. Связь между таблицами
- **328 клиентов** присутствуют в транзакциях, но отсутствуют в справочнике `clients.csv` (при `INNER JOIN` эти строки потеряются).
- **15 клиентов** есть в справочнике, но не имеют ни одной транзакции.
- Связь **one-to-many**: в среднем 104.6 транзакций на клиента (медиана — 89).

---

## 4. Итоговый план для раздела «Препроцессинг»
1. Удалить дубликаты по `client_id` и `txn_id`.
2. Конвертировать типы: даты в `datetime64`, `amount` в `float64`, `txn_id` в `string`, категориальные признаки в `category`.
3. Заменить `-999` в `monthly_income` на `NaN`.
4. Привести `city` к единому регистру (например, заглавная первая буква).
5. Обработать пропуски в `monthly_income`, `app_last_login` и `satisfaction_score` (с проверкой гипотез о механизме пропусков).

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

### 3.2. Категориальные признаки

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")